In [1]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import BertModel, BertTokenizer
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm.auto import tqdm
import os

# --- 配置路径 ---
LOCAL_BERT_PATH = "/mnt/workspace/FRD-CLIP/models/bert-base-chinese"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. 模型定义 ---
class BERT_ResNet_Baseline(nn.Module):
    def __init__(self, bert_path=LOCAL_BERT_PATH):
        super(BERT_ResNet_Baseline, self).__init__()
        # 文本部分：BERT
        self.bert = BertModel.from_pretrained(bert_path)
        
        # 图像部分：ResNet50
        # 使用预训练权重，去掉最后的 FC 层
        resnet = models.resnet50(pretrained=True)
        self.visual_backbone = nn.Sequential(*list(resnet.children())[:-1]) 
        
        # 融合层：拼接后的维度是 768 (BERT) + 2048 (ResNet50)
        self.classifier = nn.Sequential(
            nn.Linear(768 + 2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        # 提取文本特征 [batch, 768]
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_features = bert_outputs.pooler_output 
        
        # 提取图像特征 [batch, 2048]
        visual_features = self.visual_backbone(pixel_values)
        visual_features = torch.flatten(visual_features, 1)
        
        # 简单拼接
        combined = torch.cat((text_features, visual_features), dim=1)
        
        # 分类
        logits = self.classifier(combined)
        return logits

In [2]:
# --- 2. 专用 Dataset ---
# 注意：ResNet 的预处理与 CLIP 不同，需使用标准的 ImageNet 归一化
class SimpleFusionDataset(Dataset):
    def __init__(self, df, bert_tokenizer, max_len=128):
        self.df = df
        self.tokenizer = bert_tokenizer
        self.max_len = max_len
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text_clean"]) if pd.notna(row["text_clean"]) else ""
        img_path = row["img_local_path"]
        
        # 文本处理
        enc = self.tokenizer(text, padding='max_length', truncation=True, 
                             max_length=self.max_len, return_tensors="pt")
        
        # 图像处理
        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = self.transform(image)
        except:
            pixel_values = torch.zeros((3, 224, 224))
            
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values": pixel_values,
            "label": torch.tensor(int(row["label"]), dtype=torch.long)
        }

In [5]:
import copy

def train_model():
    # 1. 初始化组件
    tokenizer = BertTokenizer.from_pretrained(LOCAL_BERT_PATH)
    
    # 2. 加载数据
    train_df = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/train_ready.csv')
    val_df   = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/val_ready.csv')
    test_df  = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/test_ready.csv')
    
    train_loader = DataLoader(SimpleFusionDataset(train_df, tokenizer), batch_size=32, shuffle=True, num_workers=0)
    val_loader   = DataLoader(SimpleFusionDataset(val_df, tokenizer), batch_size=32, shuffle=False)
    test_loader  = DataLoader(SimpleFusionDataset(test_df, tokenizer), batch_size=32, shuffle=False)

    model = BERT_ResNet_Baseline().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    save_path = "best_bert_resnet_baseline.pth"

    # 3. 训练循环
    for epoch in range(5): # 增加到5个Epoch观察波动
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
            optimizer.zero_grad()
            outputs = model(batch["input_ids"].to(DEVICE), 
                            batch["attention_mask"].to(DEVICE), 
                            batch["pixel_values"].to(DEVICE))
            loss = criterion(outputs, batch["label"].to(DEVICE))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # 4. 验证阶段
        val_loss, val_acc, _, _ = evaluate_logic(model, val_loader, criterion, DEVICE)
        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # 5. 保存最优模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, save_path)
            print(f"---> Best Model Saved (Acc: {best_val_acc:.4f})")

    # 6. 最终测试
    print("\n" + "="*30 + " Training Finished " + "="*30)
    model.load_state_dict(torch.load(save_path))
    evaluate_on_test_set(model, test_loader, criterion, DEVICE)

In [6]:
from sklearn.metrics import classification_report, accuracy_score

@torch.no_grad()
def evaluate_logic(model, dataloader, criterion, device):
    """用于训练过程中的轻量化验证"""
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    for batch in dataloader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        imgs = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(ids, mask, imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, all_labels, all_preds

@torch.no_grad()
def evaluate_on_test_set(model, test_loader, criterion, device):
    """用于最后输出详细的测试报告"""
    _, acc, labels, preds = evaluate_logic(model, test_loader, criterion, device)
    
    print(f"\nFinal Results on Test Set:")
    print(f"Test Acc: {acc:.6f}")
    print("-" * 60)
    # 输出 precision, recall, f1 等
    print(classification_report(labels, preds, target_names=['real', 'fake'], digits=6))

In [7]:
train_model() # 取消注释运行

/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 [Train]:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 1: Train Loss: 0.2739 | Val Loss: 0.1724 | Val Acc: 0.9349
---> Best Model Saved (Acc: 0.9349)


Epoch 2 [Train]:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 2: Train Loss: 0.1560 | Val Loss: 0.1858 | Val Acc: 0.9170


...

Epoch 3: Train Loss: 0.1109 | Val Loss: 0.1665 | Val Acc: 0.9401
---> Best Model Saved (Acc: 0.9401)


Epoch 4 [Train]:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 4: Train Loss: 0.0742 | Val Loss: 0.1945 | Val Acc: 0.9409
---> Best Model Saved (Acc: 0.9409)


Epoch 5 [Train]:   0%|          | 0/171 [00:00<?, ?it/s]

Epoch 5: Train Loss: 0.0463 | Val Loss: 0.2424 | Val Acc: 0.9366

============================== Training Finished ==============================

Final Results on Test Set:
Test Acc: 0.932421
------------------------------------------------------------
              precision    recall  f1-score   support

        real   0.912913  0.966614  0.938996       629
        fake   0.958250  0.892593  0.924257       540

    accuracy                       0.932421      1169
   macro avg   0.935582  0.929603  0.931627      1169
weighted avg   0.933856  0.932421  0.932188      1169

